In [421]:
import numpy as np
import numba as nb
import json
import os
import subprocess

The nematic order, $Q_{ij}$, is given by

$$Q_{ij} = S(n_i n_j - \frac{1}{2}).$$

$n_i$ is a unit vector. Thus, $Q_{ij}$ is traceless and symmetric, giving it 2 degrees of freedom,

$$Q_{ij} = \begin{pmatrix}
Q_{xx} & Q_{xy}  \\
Q_{xy} & -Q_{xx} 
\end{pmatrix}.$$

$$Q_{xx} = S(n_x^2 - 1/2),$$
$$Q_{xy} = S(n_x n_y).$$

The time evolution of $Q_{ij}$ is given by

$$\partial_t Q_{ij} = -U_k \partial_k Q_{ij} + \lambda (E_{ij} - 2Q_{ij}[Q:E]) + [Q, \omega] + \Gamma H_{ij}$$

$$ H_{ij} = - \frac{\delta F_{\mathrm{LdG}}}{\partial Q_{ij}} = - A Q_{ij} (1 - 2 \mathrm{Tr}[Q^2]) + K \partial_k^2 Q_{ij}.$$

From its form, we see $H_{ij}$ is also traceless and symmetric.

$$\mathrm{Tr}[Q^2] = \mathrm{Tr}\left[ \begin{pmatrix}Q_{xx}&Q_{xy}\\ Q_{xy}&-Q_{xx}\end{pmatrix}\begin{pmatrix}Q_{xx}&Q_{xy}\\ Q_{xy}&-Q_{xx}\end{pmatrix}\right] = \mathrm{Tr}\left[\begin{pmatrix}Q_{xx}^2 + Q_{xy}^2 & 0 \\ 0 & Q_{xx}^2 + Q_{xy}^2 \end{pmatrix}\right] = 2(Q_{xx}^2 + Q_{xy}^2).$$

Strain rate and vorticity tensors are given by

$$E_{ij} = \frac{1}{2}[\partial_i u_j + \partial_j u_i]$$

and 

$$\omega_{ij} = \frac{1}{2}[\partial_i u_j - \partial_j u_i].$$

$E_{ij}$ is symmetric and $\omega_{ij}$ is antisymmetric. Incompressibility imposes that $E_{ij}$ is traceless.

Thus strain rate has 2 degrees of freedom,

$$E_{ij} = \begin{pmatrix}
E_{xx} & E_{xy} \\
E_{xy} & -E_{xx}
\end{pmatrix}.$$

$$E_{xx} = \partial_x U_x,$$
$$E_{xy} = (\partial_x U_y + \partial_y U_x)/2,$$
$$[Q : E] = \mathrm{Tr}\left[ \begin{pmatrix}Q_{xx}&Q_{xy}\\ Q_{xy}&-Q_{xx}\end{pmatrix}\begin{pmatrix}E_{xx}&E_{xy}\\ E_{xy}&-E_{xx}\end{pmatrix}\right] = \mathrm{Tr}\left[\begin{pmatrix}Q_{xx}E_{xx} + Q_{xy}E_{xy} & 0 \\ 0 & Q_{xx}E_{xx} + Q_{xy}E_{xy} \end{pmatrix}\right] = 2(Q_{xx}E_{xx} + Q_{xy}E_{xy}).$$

Vorticity only has 1 degree of freedom,

$$\omega_{ij} = \begin{pmatrix}
0 & \omega_{xy} \\
-\omega_{xy} & 0 
\end{pmatrix}.$$

$$\omega_{xy} = (\partial_x U_y - \partial_y U_x)/2,$$


$$[Q, \omega] = \begin{pmatrix}Q_{xx}&Q_{xy}\\ Q_{xy}&-Q_{xx}\end{pmatrix}\begin{pmatrix}0&\omega_{xy}\\ -\omega_{xy}&0\end{pmatrix} - \begin{pmatrix}0&\omega_{xy}\\ -\omega_{xy}&0\end{pmatrix}\begin{pmatrix}Q_{xx}&Q_{xy}\\ Q_{xy}&-Q_{xx}\end{pmatrix} = \begin{pmatrix}-2Q_{xy}\omega_{xy}&2Q_{xx}\omega_{xy}\\ 2Q_{xx}\omega_{xy}&2Q_{xy}\omega_{xy}\end{pmatrix}$$

Which is traceless and symmetric.

The time evolution of the flow field $U_i$ is given by

$$\partial_t U_i = -U_k \partial_k U_i + \nu \partial_j^2 U_i + \partial_j \Pi_{ij} - \partial_i p,\\ \partial_i U_i = 0.$$
$$\Pi_{ij} = -\lambda (H_{ij} - 2 Q_{ij}[Q:H]) - \zeta Q_{ij} + [Q, H].$$

$$[Q : H] = \mathrm{Tr}\left[ \begin{pmatrix}Q_{xx}&Q_{xy}\\ Q_{xy}&-Q_{xx}\end{pmatrix}\begin{pmatrix}H_{xx}&H_{xy}\\ H_{xy}&-H_{xx}\end{pmatrix}\right] = \mathrm{Tr}\left[\begin{pmatrix}Q_{xx}H_{xx} + Q_{xy}H_{xy} & 0 \\ 0 & Q_{xx}H_{xx} + Q_{xy}H_{xy} \end{pmatrix}\right] = 2(Q_{xx}H_{xx} + Q_{xy}H_{xy}).$$

We see that $-\lambda (H_{ij} - 2 Q_{ij}[Q:H]) - \zeta Q_{ij}$ is traceless and symmetric with 2 degrees of freedom. $\Pi^S_{ij} =-\lambda H_{ij} - \zeta Q_{ij} $.

$$[Q, H] = \begin{pmatrix}Q_{xx}&Q_{xy}\\ Q_{xy}&-Q_{xx}\end{pmatrix}\begin{pmatrix}H_{xx}&H_{xy}\\ H_{xy}&-H_{xx}\end{pmatrix} - \begin{pmatrix}H_{xx}&H_{xy}\\ H_{xy}&-H_{xx}\end{pmatrix}\begin{pmatrix}Q_{xx}&Q_{xy}\\ Q_{xy}&-Q_{xx}\end{pmatrix} = \begin{pmatrix}0&2(Q_{xx}H_{xy} - Q_{xy}H_{xx})\\ -2(Q_{xx}H_{xy} - Q_{xy}H_{xx})&0\end{pmatrix}$$

Which is anti-symmetric with 1 degree of freedom. $\Pi^A_{ij} = [Q, H]$. Thus, $$\Pi_{ij} = \Pi^S_{ij} + \Pi^A_{ij}  = \begin{pmatrix}
\Pi^S_{xx} & \Pi^S_{xy} \\
\Pi^S_{xy} & -\Pi^S_{xx}
\end{pmatrix} + \begin{pmatrix}
0 & \Pi^A_{xy} \\
-\Pi^A_{xy} & 0
\end{pmatrix}$$

$$\partial_j \Pi_{ij} = \begin{pmatrix}
\partial_x \Pi_{xx} + \partial_y \Pi_{xy} \\
\partial_x \Pi_{yx} + \partial_y \Pi_{yy}
\end{pmatrix} = 
\begin{pmatrix}
\partial_x \Pi^S_{xx} + \partial_y (\Pi^S_{xy} + \Pi^A_{xy}) \\
\partial_x (\Pi^S_xy - \Pi^A_{xy}) - \partial_y \Pi^S_{xx}
\end{pmatrix}
$$


In [422]:
@nb.njit(parallel=True, fastmath=True, nogil=True)
def get_fields(dQdt, Q, Pi_S, Pi_A, U, A, K, Z, L, G, neighbors, points):

    for p in nb.prange(points):

        # coordinates of nearest neighbors and diagonal points
        xup, xdn, yup, ydn = neighbors[p]
        xup_yup, xup_ydn = neighbors[xup][2:]
        xdn_yup, xdn_ydn = neighbors[xdn][2:]
        
        # velocity gradient tensor elements.
        dUdx = U[xup] - U[xdn]
        dUdy = U[yup] - U[ydn]

        # vorticity
        W = (dUdx[1] - dUdy[0]) * 0.25

        # \nabla^2 Q
        lap_Q = (( Q[xup] + Q[xdn] + Q[yup] + Q[ydn]) * 0.666666667
                +( Q[xup_yup] + Q[xup_ydn] + Q[xdn_yup] + Q[xdn_ydn]) * 0.166666667
                -  Q[p] * 3.333333333)
        
        Qxx, Qxy = Q[p, 0], Q[p, 1]
        
        # Tr[Q^2]
        TrQ2 = 2 * (Qxx * Qxx + Qxy * Qxy)
        
        # LdG force: K \nabla^2 Q - A Q (1 - 2 Tr[Q^2])
        H = K * lap_Q - A * Q[p] * (1 - 2 * TrQ2)

        # \Gamma * H - (U  dot \nabla) Q
        dQdt[p] = G * H - (U[p, 0] * (Q[xup] - Q[xdn]) + U[p, 1] * (Q[yup] - Q[ydn])) * 0.5
        
        # [Q, omega] + \lambda (E + 2 Q (Q : E))
        TrQE = Qxx * dUdx[0] + Qxy * (dUdy[0] + dUdx[1]) * 0.5
        dQdt[p, 0] += - 2 * Qxy * W + L * (dUdx[0] * 0.5 - 2 * TrQE * Qxx)
        dQdt[p, 1] +=   2 * Qxx * W + L * ((dUdy[0] + dUdx[1]) * 0.25 - 2 * TrQE * Qxy)

        # Pi^S = -\lambda (H - 2 Q (Q : H)) - \zeta Q, Pi^A = [Q, H]
        TrQH = 2 * (Qxx * H[0] + Qxy * H[1])
        Pi_S[p] = - L * (H - 2 * TrQH * Q[p]) - Z * Q[p]
        Pi_A[p] = 2 * (Qxx * H[1] - Qxy * H[0])

The divergence of the stress tensor can be computed only after the stress tensor is updated at all points. Thus, we need two functions; one for $\partial_t Q$ and one for $\partial_j \Pi_{ij}$.

In [423]:
@nb.njit(parallel=True, fastmath=True, nogil=True)
def get_forces(dUdt, U, Pi_S, Pi_A, N, neighbors, points):
        
        for p in nb.prange(points):
                xup, xdn, yup, ydn = neighbors[p]
                xup_yup, xup_ydn = neighbors[xup][2:]
                xdn_yup, xdn_ydn = neighbors[xdn][2:]

                # gradient of stress tensor elemenets
                dPi_Sdx = Pi_S[xup] - Pi_S[xdn]
                dPi_Sdy = Pi_S[yup] - Pi_S[ydn]

                dPi_Adx = Pi_A[xup] - Pi_A[xdn]
                dPi_Ady = Pi_A[yup] - Pi_A[ydn]
                
                # \nabla dot Pi
                dUdt[p, 0] = dPi_Sdx[0] + dPi_Sdy[1] + dPi_Ady
                dUdt[p, 1] = dPi_Sdx[1] - dPi_Sdy[0] - dPi_Adx

                # - (U dot \nabla) U
                dUdt[p] -= (U[p, 0] * (U[xup] - U[xdn]) +U[p, 1] * (U[yup] - U[ydn]) )

                dUdt[p] *= 0.5

                # nu \nabla^2 U
                dUdt[p] += N * (( U[xup] + U[xdn] + U[yup] + U[ydn]) * 0.666666667
                        + ( U[xup_yup] + U[xup_ydn] + U[xdn_yup] + U[xdn_ydn]) * 0.166666667
                        -   U[p] * 3.333333333)

Pressure plays the exclusive role of maintaining the second of the Navier-Stokes equations: $\partial_i U_i = 0.$

We achieve this by exploiting the linearity of divergence; if $$\partial_i U^t_i = 0,$$ and $$\partial_i \partial_t U_i = 0,$$ then $$\partial_i (U^t_i + (\partial_t U_i) dt) = 0.$$

Suppose that before we subtract the pressure gradient, $\partial_i \partial_t \tilde U_i = a$. The   $\tilde .$  indicates this is not the full time derivative, but the part of it that does not contain $\partial_i p$.

We would like $p$ such that $$\partial_i (\partial_t \tilde U_i - \partial_i p) = 0 \to a - \partial_i^2 p = 0 \to \partial_i^2 p = a.$$

$\partial_i \partial_t \tilde U_i = a$ can be coarsely approximated to leading order in $Q_{ij}$ as $-\zeta \partial_i \partial_j Q_{ij} \propto \partial_i (n_i \partial_j n_j - \epsilon_{ijk}\epsilon_{klm} n_j\partial_l n_m)$, which is known as the splay-bend parameter. We have no need to approximate it, but we will still refer to this scalar field as "SB".

Specifically, the source term of the Poisson equation,  $\partial_i \partial_t \tilde U_i = a$ cannot be computed in parallel with the other fields since it involves gradients of $\partial_t U_i$. The solving of the Poisson equation must also then happen after the source term is calculated, and only then can the pressure gradients finally be applied. This means the entire code can operate while sweeping through coordinates at least 5 times per timestep: one for the current fields, one to find the forces, one to calculate the Poisson source, as many as required for pressure relaxation, and one for applying the pressure gradient.

In [424]:
@nb.njit(parallel=True, fastmath=True, nogil=True)
def get_source(SB, dUdt, neighbors, points):

    for p in nb.prange(points):
        xup, xdn, yup, ydn = neighbors[p]

        # \nabla dot d(\tilde U)dt
        SB[p] = (   dUdt[xup, 0] - dUdt[xdn, 0] + 
                    dUdt[yup, 1] - dUdt[ydn, 1] ) * 0.5

In [425]:
@nb.njit(parallel=True, fastmath=True, nogil=True)
def poisson_inner(P, Pc, SB, neighbors, points):

    for p in nb.prange(points):
        xup, xdn, yup, ydn = neighbors[p]
        xup_yup, xup_ydn = neighbors[xup][2:]
        xdn_yup, xdn_ydn = neighbors[xdn][2:]

        P[p] = ((Pc[xup] + Pc[xdn] + 
                 Pc[yup] + Pc[ydn] ) * 0.2 +

                (Pc[xup_yup] + Pc[xup_ydn] +
                 Pc[xdn_yup] + Pc[xdn_ydn] ) * 0.05
                 
                - SB[p] * 0.3)

In [426]:
def poisson(P, Pc, SB, neighbors, points):
    threshold = 1e-4 * max(1, np.mean(np.abs(P)))
    diff = threshold + 1

    while diff > threshold:

        Pc[:] = P
        poisson_inner(P, Pc, SB, neighbors, points)
        diff = np.max(np.abs(P-Pc))
    
    P -= np.mean(P)

In [427]:
@nb.njit(parallel=True, fastmath=True, nogil=True)
def uncompress_flow(dUdt, P, neighbors, points):
    
    for p in nb.prange(points):
        xup, xdn, yup, ydn = neighbors[p]

        dUdt[p, 0] -= (P[xup] - P[xdn]) * 0.5
        dUdt[p, 1] -= (P[yup] - P[ydn]) * 0.5

In [428]:
@nb.njit(parallel=True, fastmath=True, nogil=True)
def randomize(Q, points):
    
    for p in nb.prange(points):

        # Completely Random Director Everywhere
        phi = np.random.uniform(0, 2 * np.pi)

        # Sin wave
        phi = np.arctan(np.cos(p // 127 / 127 * 2 * np.pi)) * np.random.uniform(.9, 1.1)
       
        Q[p, 0] = 0.5 * np.cos(2 * phi)
        Q[p, 1] = 0.5 * np.sin(2 * phi)

In [429]:
def make_boundary(boundary, Lx, Ly):
    
    neighbors = []
    if boundary == "periodic":
        for p in range(Lx * Ly):
            x = p // Ly
            y = p % Ly

            xup = ((x + 1) % Lx) * Ly + y
            xdn = ((x - 1) % Lx) * Ly + y
            yup = x * Ly + (y + 1) % Ly
            ydn = x * Ly + (y - 1) % Ly
            neighbors.append((xup, xdn, yup, ydn))

    else:
        raise ValueError("Unknown boundary condition")
    return np.array(neighbors)

In [430]:
Lx = Ly      = 2**7 - 1        # system size
T            = int(5e4)        # max time steps
K            = 2**14           # elastic modulus
als          = 2               # active length scale
ncl          = 2               # nematic coherence length
G            = 2**(-10)        # rotational diffusivity
N            = 2**9            # viscosity
L            = 1               # flow alignment
boundary     = "periodic"      # boundary condition
runname      = f"instability"  # output directory

In [431]:
dt = 1/(2**3*max(G*K, N))      # time step
Z  =  K / als**2               # active stress coefficient
A  = -K / ncl**2               # de Gennes constant

In [432]:
if __name__ == "__main__":

    nb.set_num_threads(os.cpu_count() - 1)

    subprocess.run(f"mkdir -p {runname}", shell=True)
    subprocess.run(f"mkdir -p {runname}/data/", shell=True)
    json.dump(
        {
            "System Size":Lx,
            "Elasticity":K,
            "Active Length Scale":als,
            "Nematic Coherence Length":ncl,
            "Rotational Diffusivity":G,
            "Viscosity":N,
            "Flow Allignment":L,
            "Boundry Conditions":boundary,
            "Timestep":dt,
            "Active Stress": Z,
            "DeGennes Constant": A
        }, open(f"{runname}/{runname}.json", 'w'), indent = 4
    )

    neighbors = make_boundary(boundary, Lx, Ly)
    points    = len(neighbors)

    Q         = np.zeros((points, 2))  # nematic order parameter
    dQdt      = np.zeros((points, 2))  # time derivative of Q
    Pi_S      = np.zeros((points, 2))  # symmetric stress
    Pi_A      = np.zeros(points)       # antisymmetric stress
    U         = np.zeros((points, 2))  # flow
    dUdt      = np.zeros((points, 2))  # time derivative of U
    SB        = np.zeros(points)       # pressure source
    P         = np.zeros(points)       # pressure
    Pc        = np.zeros(points)       # Poisson update copy

    randomize(Q, points)

    for t in range(T):
        
        get_fields(dQdt, Q, Pi_S, Pi_A, U, A, K, Z, L, G, neighbors, points)
        get_forces(dUdt, U, Pi_S, Pi_A, N, neighbors, points)
        get_source(SB, dUdt, neighbors, points)
        poisson(P, Pc, SB, neighbors, points)
        uncompress_flow(dUdt, P, neighbors, points)
        
        Q += dQdt * dt
        U += dUdt * dt

        if t % 100 == 0: np.savez(f"{runname}/data/{t:10d}.npz", Q=Q, U=U)

    subprocess.run(f"python3 plot_2D.py {runname}", shell=True)

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.0.13.3)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_3 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex